In [3]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.compose import TransformedTargetRegressor # Imported so joblib can unpickle it safely


df = pd.read_csv("data/spectral_feature_data.csv")
target_cols = [col for col in df.columns if col.startswith("p")]
non_feature_cols = [col for col in df.columns if col.startswith('p4')]

feature_cols = [col for col in df.columns if col not in non_feature_cols]
base_spectral_columns = [col for col in feature_cols if not col.startswith("p")]
# --- Configuration ---
# Make sure these match your previous script exactly
model_types = ["plsr", "xgb", "lgbm", "rf"]
base_dir = "models" 
eval_dir = "evaluations_log"
os.makedirs(eval_dir, exist_ok=True)

# Helper function for Mean Percentage Error (MPE)
def mean_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    # Prevent division by zero
    non_zero_mask = y_true != 0
    if not np.any(non_zero_mask):
        return 0.0 # Fallback if all true values are exactly 0
    return np.mean((y_true[non_zero_mask] - y_pred[non_zero_mask]) / y_true[non_zero_mask]) * 100

# Look at the actual saved files to figure out what our targets are
all_targets = set()
for m_type in model_types:
    m_dir = os.path.join(base_dir, m_type)
    if os.path.exists(m_dir):
        for file in os.listdir(m_dir):
            if file.endswith("_model.pkl"):
                # Extract the target name from the filename
                target_name = file.replace("_model.pkl", "")
                all_targets.add(target_name)

print(f"Found targets to evaluate: {all_targets}")
print("Starting Evaluation Pipeline...")

for target in all_targets:
    print(f"\n========== EVALUATING {target} ==========")
    
    # ---------------------------------------------------------
    # 1. RECONSTRUCT THE DATASET
    # ---------------------------------------------------------
    prediction_columns = base_spectral_columns.copy() + ["p1.pH.index", "p1.EC.ds_m"]

    
    maskpH = df["p1.pH.index"].notna()
    maskEC = df["p1.EC.ds_m"].notna()
    target_mask = df[target].notna()
    
    final_mask = target_mask & maskpH & maskEC
    
    if final_mask.sum() == 0:
        print(f"⚠️ Skipping {target} - Not enough overlapping data.")
        continue
        
    y_clean = df.loc[final_mask, target]
    X_clean = df.loc[final_mask, prediction_columns]
    
    # EXACT same split
    X_train, X_test, y_train, y_test = train_test_split(
        X_clean, y_clean, test_size=0.2, random_state=42
    )
    
    # ---------------------------------------------------------
    # 2. LOAD MODELS & GENERATE PREDICTIONS
    # ---------------------------------------------------------
    test_predictions = {}
    train_cv_predictions = {}
    
    for m_type in model_types:
        file_path = os.path.join(base_dir, m_type, f"{target}_model.pkl")
        
        if not os.path.exists(file_path):
            print(f"  Missing {m_type.upper()} model for {target}, skipping this model.")
            continue
            
        # Load the saved pipeline package
        save_package = joblib.load(file_path)
        pipeline = save_package['pipeline']
        required_features = save_package['features_required']
        
        # Subset the X data to only what this specific model requires
        X_train_model = X_train[required_features]
        X_test_model = X_test[required_features]
        
        # Predict on Test Set (for final evaluation)
        test_predictions[m_type.upper()] = pipeline.predict(X_test_model)
        
        # Generate Cross-Validated Predictions on Train Set (for the Stacker)
        train_cv_predictions[m_type.upper()] = cross_val_predict(
            pipeline, X_train_model, y_train, cv=5, n_jobs=-1
        )
    
    if not test_predictions:
        print(f"  No models found for {target}. Moving to next.")
        continue

    # ---------------------------------------------------------
    # 3. TRAIN THE STACKER (META-MODEL)
    # ---------------------------------------------------------
    print("  Training Linear Stacker...")
    X_train_meta = pd.DataFrame(train_cv_predictions)
    X_test_meta = pd.DataFrame(test_predictions)
    
    # Add positive=True to force Non-Negative coefficients to prevent Stacker explosion
    stacker = LinearRegression(positive=True) 
    stacker.fit(X_train_meta, y_train)
    
    # Get final stacked predictions for the test set
    test_predictions['STACKER'] = stacker.predict(X_test_meta)
    
    print(f"  Stacker Weights:")
    for model_name, weight in zip(X_train_meta.columns, stacker.coef_):
        print(f"    - {model_name}: {weight:.4f}")

    # ---------------------------------------------------------
    # 4. CALCULATE METRICS
    # ---------------------------------------------------------
    models_to_eval = list(test_predictions.keys())
    metrics_data = []
    
    for model_name in models_to_eval:
        preds = test_predictions[model_name]
        r2 = r2_score(y_test, preds)
        mae = mean_absolute_error(y_test, preds)
        mpe = mean_percentage_error(y_test, preds)
        
        metrics_data.append({
            'Model': model_name,
            'R2': r2,
            'MAE': mae,
            'MPE': mpe
        })
        
    metrics_df = pd.DataFrame(metrics_data)
    
    # ---------------------------------------------------------
    # 5. GENERATE DASHBOARD
    # ---------------------------------------------------------
    print("  Generating Dashboard...")
    sns.set_theme(style="whitegrid")
    
    fig = plt.figure(figsize=(24, 12))
    fig.suptitle(f'Model Evaluation Dashboard: {target}', fontsize=24, fontweight='bold', y=0.98)
    
    gs = fig.add_gridspec(2, 15) 
    
    # --- TOP ROW: Scatter Plots ---
    for i, model_name in enumerate(models_to_eval):
        ax = fig.add_subplot(gs[0, i*3:(i+1)*3])
        preds = test_predictions[model_name]
        
        ax.scatter(y_test, preds, alpha=0.6, edgecolors='w', s=50)
        
        min_val = min(y_test.min(), preds.min())
        max_val = max(y_test.max(), preds.max())
        ax.plot([min_val, max_val], [min_val, max_val], 'k--', lw=2, label='1:1 Line')
        
        z = np.polyfit(y_test, preds, 1)
        p = np.poly1d(z)
        ax.plot(y_test, p(y_test), 'r-', lw=2, label='Trendline')
        
        ax.set_title(model_name, fontsize=16, fontweight='bold')
        ax.set_xlabel('Actual Values', fontsize=12)
        ax.set_ylabel('Predicted Values', fontsize=12)
        ax.legend()

    # --- BOTTOM ROW: Bar Charts ---
    ax_r2 = fig.add_subplot(gs[1, 1:5])
    sns.barplot(data=metrics_df, x='Model', y='R2', ax=ax_r2, palette='viridis')
    ax_r2.set_title('$R^2$ Score (Higher is Better)', fontsize=14)
    # Dynamically adjust Y-axis so negative R2 values are visible if a model completely bombs
    ax_r2.set_ylim(min(0, metrics_df['R2'].min() - 0.1), 1.0) 
    
    ax_mae = fig.add_subplot(gs[1, 6:10])
    sns.barplot(data=metrics_df, x='Model', y='MAE', ax=ax_mae, palette='magma')
    ax_mae.set_title('Mean Absolute Error (Lower is Better)', fontsize=14)
    
    ax_mpe = fig.add_subplot(gs[1, 11:15])
    sns.barplot(data=metrics_df, x='Model', y='MPE', ax=ax_mpe, palette='coolwarm')
    ax_mpe.set_title('Mean Percentage Error (%)', fontsize=14)
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    
    save_path = os.path.join(eval_dir, f"{target}_dashboard.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  -> Dashboard saved to {save_path}")

print("\nAll evaluations complete!")

Found targets to evaluate: {'p3.S.wt_pct', 'p3.K.mg_kg', 'p1.Sand.wt_pct', 'p2.OC.wt_pct', 'p4.WR_10kPa.wt_pct', 'p2.N.wt_pct', 'p4.WR_1500kPa.wt_pct', 'p1.Silt.wt_pct', 'p3.P.mg_kg', 'p4.WR_33kPa.wt_pct', 'p4.BD.g_cm3', 'p1.EC.ds_m', 'p1.Clay.wt_pct', 'p4.CEC.cmolc_kg', 'p1.pH.index', 'p4.CF.wt_pct'}
Starting Evaluation Pipeline...

========== EVALUATING p3.S.wt_pct ==========


c:\Users\jagda\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  Training Linear Stacker...
  Stacker Weights:
    - PLSR: 0.5666
    - XGB: 0.7999
    - LGBM: 0.0000
    - RF: 0.0000
  Generating Dashboard...


C:\Users\jagda\AppData\Local\Temp\ipykernel_53104\4076054436.py:182: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=metrics_df, x='Model', y='R2', ax=ax_r2, palette='viridis')
C:\Users\jagda\AppData\Local\Temp\ipykernel_53104\4076054436.py:188: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=metrics_df, x='Model', y='MAE', ax=ax_mae, palette='magma')
C:\Users\jagda\AppData\Local\Temp\ipykernel_53104\4076054436.py:192: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=metrics_df, x='Model', y='MPE', ax=ax_mpe, palette='coolwarm')


  -> Dashboard saved to evaluations_log\p3.S.wt_pct_dashboard.png

========== EVALUATING p3.K.mg_kg ==========


...

  -> Dashboard saved to evaluations_log\p1.pH.index_dashboard.png

========== EVALUATING p4.CF.wt_pct ==========


c:\Users\jagda\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  Training Linear Stacker...
  Stacker Weights:
    - PLSR: 0.6936
    - XGB: 0.3466
    - LGBM: 0.0000
    - RF: 0.1757
  Generating Dashboard...


C:\Users\jagda\AppData\Local\Temp\ipykernel_53104\4076054436.py:182: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=metrics_df, x='Model', y='R2', ax=ax_r2, palette='viridis')
C:\Users\jagda\AppData\Local\Temp\ipykernel_53104\4076054436.py:188: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=metrics_df, x='Model', y='MAE', ax=ax_mae, palette='magma')
C:\Users\jagda\AppData\Local\Temp\ipykernel_53104\4076054436.py:192: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=metrics_df, x='Model', y='MPE', ax=ax_mpe, palette='coolwarm')


  -> Dashboard saved to evaluations_log\p4.CF.wt_pct_dashboard.png

All evaluations complete!


In [2]:
#without log transform

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error


df = pd.read_csv("data/spectral_feature_data.csv")
target_cols = [col for col in df.columns if col.startswith("p")]
non_feature_cols = [col for col in df.columns if col.startswith('p4')]

feature_cols = [col for col in df.columns if col not in non_feature_cols]
base_spectral_columns = [col for col in feature_cols if not col.startswith("p")]

# --- Configuration ---
model_types = ["plsr", "xgb", "lgbm", "rf"]
base_dir = "models" 
# Renamed so you can compare side-by-side with the log-transformed results
eval_dir = "evaluations_base" 
os.makedirs(eval_dir, exist_ok=True)

# Helper function for Mean Percentage Error (MPE)
def mean_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    non_zero_mask = y_true != 0
    if not np.any(non_zero_mask):
        return 0.0 
    return np.mean((y_true[non_zero_mask] - y_pred[non_zero_mask]) / y_true[non_zero_mask]) * 100

# Robust target discovery via directory scanning
all_targets = set()
for m_type in model_types:
    m_path = os.path.join(base_dir, m_type)
    if os.path.exists(m_path):
        for f in os.listdir(m_path):
            if f.endswith("_model.pkl"):
                all_targets.add(f.replace("_model.pkl", ""))

print(f"Found {len(all_targets)} targets to evaluate in Base Case...")

for target in all_targets:
    print(f"\n========== EVALUATING {target} (BASE CASE) ==========")
    
    # 1. Dataset Reconstruction
    # Assuming base_spectral_columns is defined in your environment
    prediction_columns = base_spectral_columns.copy() + ["p1.pH.index", "p1.EC.ds_m"]
    
    maskpH = df["p1.pH.index"].notna()
    maskEC = df["p1.EC.ds_m"].notna()
    target_mask = df[target].notna()
    final_mask = target_mask & maskpH & maskEC
    
    if final_mask.sum() < 10: # Safety check for small samples
        print(f"⚠️ Skipping {target} - Insufficient data ({final_mask.sum()} rows).")
        continue
        
    y_clean = df.loc[final_mask, target]
    X_clean = df.loc[final_mask, prediction_columns]
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_clean, y_clean, test_size=0.2, random_state=42
    )
    
    # 2. Load Models & Inference
    test_predictions = {}
    train_cv_predictions = {}
    
    for m_type in model_types:
        file_path = os.path.join(base_dir, m_type, f"{target}_model.pkl")
        if not os.path.exists(file_path): continue
            
        save_package = joblib.load(file_path)
        pipeline = save_package['pipeline']
        req_features = save_package['features_required']
        
        test_predictions[m_type.upper()] = pipeline.predict(X_test[req_features])
        
        # Cross-val for the Stacker
        train_cv_predictions[m_type.upper()] = cross_val_predict(
            pipeline, X_train[req_features], y_train, cv=5, n_jobs=-1
        )

    # 3. Train Stacker (Meta-Model)
    X_train_meta = pd.DataFrame(train_cv_predictions)
    X_test_meta = pd.DataFrame(test_predictions)
    
    # Stacker Fix: Positive=True to handle multicollinearity
    stacker = LinearRegression(positive=True) 
    stacker.fit(X_train_meta, y_train)
    test_predictions['STACKER'] = stacker.predict(X_test_meta)
    
    # 4. Metrics & Plotting
    metrics_data = []
    for m_name, preds in test_predictions.items():
        metrics_data.append({
            'Model': m_name,
            'R2': r2_score(y_test, preds),
            'MAE': mean_absolute_error(y_test, preds),
            'MPE': mean_percentage_error(y_test, preds)
        })
    metrics_df = pd.DataFrame(metrics_data)
    
    # 5. Dashboard Generation
    sns.set_theme(style="whitegrid")
    fig = plt.figure(figsize=(24, 12))
    fig.suptitle(f'BASE CASE Evaluation: {target}', fontsize=24, fontweight='bold', y=0.98)
    gs = fig.add_gridspec(2, 15) 
    
    models_sorted = sorted(test_predictions.keys())
    for i, m_name in enumerate(models_sorted):
        ax = fig.add_subplot(gs[0, i*3:(i+1)*3])
        p = test_predictions[m_name]
        ax.scatter(y_test, p, alpha=0.6, edgecolors='w')
        
        # 1:1 and Trendline
        lims = [np.min([y_test.min(), p.min()]), np.max([y_test.max(), p.max()])]
        ax.plot(lims, lims, 'k--', alpha=0.75, zorder=0, label='1:1')
        z = np.polyfit(y_test, p, 1)
        ax.plot(y_test, np.poly1d(z)(y_test), 'r-', label='Trend')
        
        ax.set_title(m_name, fontweight='bold')
        ax.legend()

    # Bar Charts
    ax_r2 = fig.add_subplot(gs[1, 1:5])
    sns.barplot(data=metrics_df, x='Model', y='R2', ax=ax_r2, palette='viridis')
    ax_r2.set_ylim(min(0, metrics_df['R2'].min()), 1.0)
    
    ax_mae = fig.add_subplot(gs[1, 6:10])
    sns.barplot(data=metrics_df, x='Model', y='MAE', ax=ax_mae, palette='magma')
    
    ax_mpe = fig.add_subplot(gs[1, 11:15])
    sns.barplot(data=metrics_df, x='Model', y='MPE', ax=ax_mpe, palette='coolwarm')
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(os.path.join(eval_dir, f"{target}_base_dashboard.png"), dpi=300)
    plt.close()

print("\nBase Case Dashboards completed.")

Found 16 targets to evaluate in Base Case...

========== EVALUATING p3.K.mg_kg (BASE CASE) ==========


c:\Users\jagda\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\jagda\AppData\Local\Temp\ipykernel_29712\4293811620.py:132: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=metrics_df, x='Model', y='R2', ax=ax_r2, palette='viridis')
C:\Users\jagda\AppData\Local\Temp\ipykernel_29712\4293811620.py:136: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=metrics_df, x='Model', y='MAE', ax=ax_mae, palette='magma')
C:\Users\jagda\AppData\Local\Temp\ipykernel_29712\4293811620.py:139: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and 


========== EVALUATING p1.pH.index (BASE CASE) ==========


c:\Users\jagda\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\jagda\AppData\Local\Temp\ipykernel_29712\4293811620.py:132: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=metrics_df, x='Model', y='R2', ax=ax_r2, palette='viridis')
C:\Users\jagda\AppData\Local\Temp\ipykernel_29712\4293811620.py:136: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=metrics_df, x='Model', y='MAE', ax=ax_mae, palette='magma')
C:\Users\jagda\AppData\Local\Temp\ipykernel_29712\4293811620.py:139: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and 


========== EVALUATING p3.P.mg_kg (BASE CASE) ==========


...


========== EVALUATING p3.S.wt_pct (BASE CASE) ==========


c:\Users\jagda\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\jagda\AppData\Local\Temp\ipykernel_29712\4293811620.py:132: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=metrics_df, x='Model', y='R2', ax=ax_r2, palette='viridis')
C:\Users\jagda\AppData\Local\Temp\ipykernel_29712\4293811620.py:136: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=metrics_df, x='Model', y='MAE', ax=ax_mae, palette='magma')
C:\Users\jagda\AppData\Local\Temp\ipykernel_29712\4293811620.py:139: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and 


========== EVALUATING p4.WR_10kPa.wt_pct (BASE CASE) ==========
⚠️ Skipping p4.WR_10kPa.wt_pct - Insufficient data (0 rows).

========== EVALUATING p4.CEC.cmolc_kg (BASE CASE) ==========


c:\Users\jagda\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\jagda\AppData\Local\Temp\ipykernel_29712\4293811620.py:132: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=metrics_df, x='Model', y='R2', ax=ax_r2, palette='viridis')
C:\Users\jagda\AppData\Local\Temp\ipykernel_29712\4293811620.py:136: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=metrics_df, x='Model', y='MAE', ax=ax_mae, palette='magma')
C:\Users\jagda\AppData\Local\Temp\ipykernel_29712\4293811620.py:139: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and 


Base Case Dashboards completed.
